In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [2]:
llm = ChatOpenAI(model="gpt-5.4-mini")
question= "what is the meaning of life"
answer= llm.invoke(question)
print(answer.content)

There isn’t one single universally agreed “meaning of life.”

A few common answers are:

- **Biological:** to survive, reproduce, and pass on life.
- **Philosophical:** to create meaning through your choices, values, and relationships.
- **Relational:** to love, be loved, and connect with others.
- **Personal:** to pursue what matters most to you—growth, service, creativity, knowledge, faith, joy.

A practical answer is: **the meaning of life is the meaning you build**.

If you want, I can also answer this from a **religious**, **philosophical**, or **scientific** perspective.


# My Learning
# Four LangChain Keywords

`ChatOpenAI` is a LangChain class that connects your Python code to an OpenAI chat model. It creates an LLM object that you can use to ask questions and get responses. For example, `llm = ChatOpenAI(model="gpt-5.4-mini")`.

`model` is an argument passed into `ChatOpenAI`. It tells LangChain which OpenAI model you want to use. In `ChatOpenAI(model="gpt-5.4-mini")`, the selected model is `"gpt-5.4-mini"`.

`invoke()` is a LangChain method used to send input to the model. It takes your prompt or question and returns the model’s response. For example, `answer = llm.invoke("What is the meaning of life?")`.

`content` contains the actual text answer from the model. `llm.invoke()` returns a response object, not just plain text. The `.content` part extracts the model’s written response. For example, `print(answer.content)` prints only the response text.

Example:

```python
llm = ChatOpenAI(model="gpt-5.4-mini")
question = "what is the meaning of life"
answer = llm.invoke(question)
print(answer.content)

In [3]:
for chunk in llm.stream("Tell me about two autonomous agents"):
    print (chunk.content, end="", flush=True)

Two autonomous agents are systems that can operate on their own to pursue goals, make decisions, and take actions with limited or no human intervention.

A simple way to think about them:

- **Agent 1:** Observes its environment, decides what to do, acts, and repeats.
- **Agent 2:** Does the same, possibly with different goals, tools, or rules.

## Common examples
1. **Self-driving cars**
   - Sense roads, traffic, and obstacles
   - Decide when to speed up, slow down, or turn
   - Act in real time without human control

2. **Trading bots**
   - Monitor market conditions
   - Decide when to buy or sell
   - Execute trades automatically

## Typical components of an autonomous agent
- **Sensors / input**: What it can observe
- **Decision logic**: Rules, models, or planning
- **Action system**: What it can do
- **Memory / state**: What it remembers over time
- **Goal**: What it is trying to achieve

## Why they matter
- Can work continuously
- Can react faster than humans
- Useful in envi

# My Learning
## Use of `stream()` in Production Agents

`stream()` is used to receive the model’s response little by little instead of waiting for the full answer.

This is useful in production agents because it makes the app feel faster and more responsive.

Example:

```python
for chunk in llm.stream("Tell me about two autonomous agents"):
    print(chunk.content, end="", flush=True)

In [2]:
# openrouter_llm= ChatOpenAI(
#     model="anthropic/claude-haiku-4.5",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPENROUTER_API_KEY"),
# )

# reply= openrouter_llm.invoke("Explain me langchain in one sentence")
# print(reply.content)

## OpenRouter with LangChain

OpenRouter lets you access models from different providers like Anthropic, OpenAI, Google, Meta, and Mistral through one API. In LangChain, OpenRouter is usually used with `ChatOpenAI` because OpenRouter provides an OpenAI-compatible API.

Even if you use an Anthropic model through OpenRouter, you still use `ChatOpenAI`, not `ChatAnthropic`.

Example:

`openrouter_llm = ChatOpenAI(model="anthropic/claude-haiku-4.5", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))`

`reply = openrouter_llm.invoke("In one sentence, what is LangChain?")`

`print(reply.content)`

Simple rule: Anthropic direct API uses `ChatAnthropic`, but Anthropic through OpenRouter uses `ChatOpenAI`.

OpenRouter is useful in production because it allows easy model switching, fallback models, cost optimization, and access to many providers through one LangChain interface.

In [4]:
messages = [
    SystemMessage("you are a travel guide, and you help people to setup travel itenaries for the place they mention"),
    HumanMessage("help me plan a trip in puerto rico from austin for 5 days mid level budget"),
]

# print(llm.invoke(messages).content)
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)

Absolutely — here’s a solid **5-day mid-budget Puerto Rico trip from Austin**. I’ll assume you want a mix of **beach, Old San Juan, food, and maybe one adventure day** while keeping costs reasonable.

## Quick trip overview
- **Destination:** Puerto Rico
- **Departure:** Austin (AUS)
- **Trip length:** 5 days
- **Budget level:** Mid-range
- **Best home base:** **San Juan** for easy access to sightseeing, food, and day trips

---

## Estimated budget per person
A mid-budget 5-day trip from Austin to Puerto Rico could look like this:

- **Flights:** $250–$500 roundtrip
- **Hotel/Airbnb:** $120–$220 per night
- **Food:** $40–$80 per day
- **Transportation:** $30–$100 total if using Ubers/public transit, more if renting a car
- **Activities:** $100–$250 total

**Estimated total:** **$900–$1,800 per person** depending on flight prices and lodging choice

---

## Suggested 5-day itinerary

### Day 1: Fly to San Juan + Old San Juan evening
**Morning/Afternoon**
- Fly from Austin to **Luis Muñ

## LangChain Messages

Every turn in a conversation with a chat model is represented as a `BaseMessage` subclass.

| Class | Role | Written by | Purpose |
|---|---|---|---|
| `SystemMessage` | `system` | you | Sets the model's behavior/persona/instructions |
| `HumanMessage` | `user` | you | The user's input |
| `AIMessage` | `assistant` | the model | The model's response — either a real answer, or a request to call a tool |
| `ToolMessage` | `tool` | you | The result of running a tool, reported back to the model |

### AIMessage
Returned every time you call `llm.invoke(...)`. Two shapes:
- **Plain answer:** `AIMessage(content="Paris is the capital of France.")`
- **Tool request:** `AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "Paris"}, "id": "call_123"}])`

The model never runs the tool itself — it only requests it.

### ToolMessage
Your code executes the actual tool (a Python function, API call, etc.), then packages the result as a `ToolMessage` so the model can continue:

```python
ToolMessage(content="72°F and sunny", tool_call_id="call_123")
```

`tool_call_id` must match the `id` from the corresponding tool call in the `AIMessage`, so the model knows which request the result answers.

### Full round trip

```python
messages = [HumanMessage(content="What's the weather in Paris?")]

# Turn 1: model requests a tool instead of answering directly
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

# Your code runs the actual tool
result = get_weather(city="Paris")  # -> "72°F and sunny"
messages.append(ToolMessage(content=result, tool_call_id=ai_msg.tool_calls[0]["id"]))

# Turn 2: model now has what it needs to answer
final = llm_with_tools.invoke(messages)
print(final.content)  # "It's 72°F and sunny in Paris."
```

**Summary:** `AIMessage` = the model's turn (answer or tool request). `ToolMessage` = your turn, reporting back what the tool returned. Messages are provider-agnostic — the same classes work with `ChatAnthropic`, `ChatOpenAI`, or any other LangChain chat model.


In [5]:
@tool
def get_share_prices(symbol: str) ->float:
    "Return share prices for the given symbol"
    prices= {"AAPL":141.5, "IBM":232.5, "AMZN":196.5, "NVDA":199 }
    return prices.get(symbol.upper(), 0.0)

print (get_share_prices.invoke("AAPL"))

141.5


In [6]:
@tool
def get_company_share_value_avg(symbol: str) -> float:
    "Tool that returns company's average sharing price for MTD"
    prices = {
        "AAPL": [140.2, 141.0, 141.5, 142.8, 140.9],
        "IBM": [230.1, 231.5, 232.5, 233.0, 229.8],
        "AMZN": [194.0, 195.2, 196.5, 197.1, 195.8],
        "NVDA": [197.5, 198.2, 199.0, 200.1, 198.8],
    }
    daily_prices = prices.get(symbol.upper())
    if not daily_prices:
        return 0.0
    return sum(daily_prices) / len(daily_prices)

## LangChain Tools & `bind_tools`

### `@tool` — defining a tool
The `@tool` decorator turns a plain Python function into something an LLM can be given access to and choose to call. The function still runs as normal Python code — the decorator just wraps it so LangChain can describe it to the model.

Two things matter most when defining a tool:
- **The docstring is required**, and becomes the tool's description. This is how the model decides *when* it's appropriate to call this tool, based on the user's request.
- **The type hints on the parameters** define the tool's input schema — they tell the model exactly what arguments to provide and what type each one should be.

Once wrapped, the function is no longer called like an ordinary Python function. It becomes a tool object, invoked through a standard interface rather than direct calling — which lets you manually test the tool's own logic before ever connecting it to a model.

### `bind_tools` — giving the model access to tools
`bind_tools` attaches one or more tools to a chat model. Importantly, it doesn't execute anything by itself — it simply informs the model which tools exist, along with their names, descriptions, and expected arguments.

The model still cannot run a tool on its own. When it decides a tool is needed to answer a question, it doesn't return a final answer — instead, it returns a response indicating which tool it wants to use and with what arguments. It's then up to your own code to actually execute that tool.

Once your code runs the tool, the result is sent back to the model as its own kind of message, so the model can incorporate that result into a final, human-readable answer.

### The overall flow
1. A tool is defined and described using `@tool`.
2. The tool is attached to the model using `bind_tools`, making it "available" but not yet used.
3. The model, upon receiving a question, decides whether it needs a tool and requests one instead of answering directly.
4. Your code executes the requested tool with the arguments the model provided.
5. The result is handed back to the model.
6. The model uses that result to produce the final answer to the original question.

This separation is important: the model only ever *decides and requests* — it never directly executes code. Execution is always your application's responsibility, which keeps tool use safe and controllable.


In [7]:
llm_with_tool = llm.bind_tools([get_share_prices, get_company_share_value_avg])

response = llm_with_tool.invoke("what is the share price of Amazon and month to date value?")

print("content:", (response.content))
print ("tool_answer:",(response.tool_calls))

content: 
tool_answer: [{'name': 'get_share_prices', 'args': {'symbol': 'AMZN'}, 'id': 'call_F7b6U9CBx1M18lF3tg2lmNpp', 'type': 'tool_call'}, {'name': 'get_company_share_value_avg', 'args': {'symbol': 'AMZN'}, 'id': 'call_QPYeDjHZ5iEuzBtCEkozjtpr', 'type': 'tool_call'}]


In [8]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("what is the share price of Apple and month to date value?")]
ai_message = llm_with_tool.invoke(conversation)
conversation.append(ai_message)

available_tools = [
    get_share_prices,
    get_company_share_value_avg,
    # add more tools here
]
print("tool_calls:", ai_message.tool_calls)
tool_registry = {tool.name: tool for tool in available_tools}

for call in ai_message.tool_calls:
    tool_name = call["name"]
    tool = tool_registry.get(tool_name)

    if tool is None:
        result = f"Unknown tool: {tool_name}"
    else:
        try:
            result = tool.invoke(call.get("args", {}))
        except Exception as e:
            result = f"Error running tool {tool_name}: {type(e).__name__}: {e}"

    conversation.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"],
        )
    )

# Invoke again, now that the model can see the tool result
final = llm_with_tool.invoke(conversation)
print(final.content)

tool_calls: [{'name': 'get_share_prices', 'args': {'symbol': 'AAPL'}, 'id': 'call_UshK84sNTt9SQ6vUrNUicLOz', 'type': 'tool_call'}, {'name': 'get_company_share_value_avg', 'args': {'symbol': 'AAPL'}, 'id': 'call_AANNQC6bC1LX04xhsdXTJDxN', 'type': 'tool_call'}]
Apple (AAPL):
- Share price: 141.50
- Month-to-date average share value: 141.28




In [32]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN


## `bind_tools` and Why We Build the Answer Ourselves

`bind_tools` tells the AI which tools it's allowed to use — it doesn't run anything, it just makes the AI *aware* that these tools exist.

```python
llm_with_tools = llm.bind_tools([get_share_prices])
```

When we ask a question, the AI can't actually fetch real data itself — it can only think and write text. So instead of answering right away, it responds with `tool_calls`, which is just its way of saying **"here's what I want you to look up for me."**

`response.tool_calls` is **not the answer** — it's only the AI's plan for how it wants to proceed. It tells us:
- which tool it wants used
- what arguments to use
- an ID to match the result back to this request

Since the AI can't run the tool itself, **we have to manually**:
1. Run the real tool with the arguments the AI gave us
2. Wrap the result in a `ToolMessage`
3. Send the whole conversation back to the AI

Only after we've done that does the AI give us a real, final answer — the actual number or sentence never comes from `tool_calls` itself. It only comes after we complete these manual steps and ask the AI again. (Just a manual non-dynamic method)


# AgentEvals

There are two types of evaluations in **AgentEvals**:

1. **Trajectory Match**
2. **LLM-as-a-Judge**

## Trajectory Match

Trajectory matching supports four matching modes:

- **Strict:** The message structure and tool calls must exactly match the reference and occur in the same order. Message content may differ.

- **Unordered:** The message structure and tool calls must match the reference, but tool calls may occur in any order.

- **Subset:** The agent may call only tools included in the reference trajectory. No additional tools are allowed.

- **Superset:** The agent must call at least the tools included in the reference trajectory. Additional tool calls are allowed.

In [15]:
from agentevals.trajectory.match import create_trajectory_match_evaluator
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

question = "what is the share price of Amazon and month to date value?"

# `response` is the AIMessage containing the two tool calls.
share_price_result = get_share_prices.invoke(response.tool_calls[0]["args"])
month_to_date_result = get_company_share_value_avg.invoke(
    response.tool_calls[1]["args"]
)

# Keep the tool-call response, tool results, and final answer in one trajectory.
actual_trajectory = [
    HumanMessage(content=question),
    response,
    ToolMessage(
        content=str(share_price_result),
        tool_call_id=response.tool_calls[0]["id"],
    ),
    ToolMessage(
        content=str(month_to_date_result),
        tool_call_id=response.tool_calls[1]["id"],
    ),
]

final_response = llm_with_tool.invoke(actual_trajectory)
actual_trajectory.append(final_response)

# The tool calls we expect for this question.
reference_trajectory = [
    HumanMessage(content=question),
    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "get_share_prices",
                "args": {"symbol": "AMZN"},
                "id": "expected-price-call",
                "type": "tool_call",
            },
            {
                "name": "get_company_share_value_avg",
                "args": {"symbol": "AMZN"},
                "id": "expected-month-to-date-call",
                "type": "tool_call",
            },
        ],
    ),
    ToolMessage(
        content="196.5",
        tool_call_id="expected-price-call",
    ),
    ToolMessage(
        content="195.72",
        tool_call_id="expected-month-to-date-call",
    ),
    AIMessage(
        content="Amazon's share price is $196.50 and its month-to-date average is $195.72."
    ),
]


In [16]:
# Strict: same message structure and tool-call order
strict_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict"
)

strict_result = strict_evaluator(
    outputs=actual_trajectory,
    reference_outputs=reference_trajectory,
)
strict_result


{'key': 'trajectory_strict_match',
 'score': True,
 'comment': None,
 'metadata': None}

In [17]:
# Unordered: same tool calls, but their order may differ
unordered_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="unordered"
)

unordered_result = unordered_evaluator(
    outputs=actual_trajectory,
    reference_outputs=reference_trajectory,
)
unordered_result


{'key': 'trajectory_unordered_match',
 'score': True,
 'comment': None,
 'metadata': None}

In [18]:
# Subset: actual tool calls must be a subset of the reference calls
subset_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="subset"
)

subset_result = subset_evaluator(
    outputs=actual_trajectory,
    reference_outputs=reference_trajectory,
)
subset_result


{'key': 'trajectory_subset_match',
 'score': True,
 'comment': None,
 'metadata': None}

In [19]:
# Superset: actual trajectory must include all reference tool calls
superset_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="superset"
)

superset_result = superset_evaluator(
    outputs=actual_trajectory,
    reference_outputs=reference_trajectory,
)
superset_result


{'key': 'trajectory_superset_match',
 'score': True,
 'comment': None,
 'metadata': None}

LLM as judge evaluator

this has two types:

Without reference trajectory
with reference trajectory

In [20]:
# LLM-as-judge without a reference trajectory
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT,
)

judge_without_reference = create_trajectory_llm_as_judge(
    model="openai:o3-mini",
    prompt=TRAJECTORY_ACCURACY_PROMPT,
)

without_reference_result = judge_without_reference(
    outputs=actual_trajectory,
)
without_reference_result


{'key': 'trajectory_accuracy',
 'score': True,
 'comment': "The trajectory demonstrates a clear and logical progression towards the goal. The first step involves parsing the user's request by identifying the need for both the current share price and the month-to-date share value. It then appropriately makes two tool calls to retrieve each piece of information separately. The results are correctly matched with the correct API responses, and the final answer is constructed accurately. The trajectory meets the criteria for having logical progression, clear steps, and relative efficiency in answering the question. Thus, the score should be: true.",
 'metadata': None}

In [21]:
# LLM-as-judge with a reference trajectory
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
)

judge_with_reference = create_trajectory_llm_as_judge(
    model="openai:o3-mini",
    prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
)

with_reference_result = judge_with_reference(
    outputs=actual_trajectory,
    reference_outputs=reference_trajectory,
)
with_reference_result


{'key': 'trajectory_accuracy',
 'score': True,
 'comment': 'The actual trajectory contains the same sequence of actions as the reference trajectory. Both trajectories begin with the same user query, make tool calls to retrieve share price and month-to-date value, and finally provide a response that is semantically equivalent to the response in the reference trajectory. The steps are logical, progressive, and efficient. Thus, the score should be: true.',
 'metadata': None}